# ETF Risk Metrics

This notebook calculates the historical market-risk measures used in the ETF risk framework, including returns, volatility, maximum drawdown, beta, Value at Risk and Conditional Value at Risk.

In [1]:
import numpy as np
import pandas as pd

clean_prices = pd.read_csv(
    "../data/processed/etf_prices_clean.csv",
    parse_dates=["Date"],
    index_col="Date"
)

clean_prices.head()

,VWRP.L,SWDA.L,VUAG.L,ISF.L,EQQQ.L,VEUR.L,VJPN.L,VFEG.L,IUIT.L,IUHC.L,INRG.L,VGOV.L,IGLO.L,IBTG.L,AGGG.L,CORP.L,GHYS.L,SGLN.L,IWDP.L,ICHN.AS
Date,,,,,,,,,,,,,,,,,,,,
2021-07-22,81.834999,6083.5,57.724998,681.851929,26504.806641,26.007370,23.586449,48.197498,19.195000,9.78750,950.598511,21.207935,105.799774,4.321871,4.794285,90.257408,76.450050,2565.0,2121.746582,7.0221
2021-07-23,82.419998,6141.0,58.334999,687.840942,26790.751953,26.219423,23.677200,47.404999,19.400000,9.90375,937.104309,21.170839,105.571640,4.322741,4.787110,90.173080,76.567162,2555.0,2125.741211,6.7538
2021-07-26,82.040001,6123.0,58.205002,687.641296,26751.763672,26.174442,23.393597,46.355000,19.392500,9.86125,927.358459,21.194160,105.945778,4.322306,4.799667,90.316452,76.590599,2540.5,2117.503662,6.4750
2021-07-27,80.830002,6053.0,57.605000,683.947998,26160.878906,26.022366,23.273354,44.840000,19.084999,9.88375,906.367310,21.256691,106.246918,4.322741,4.808637,90.476616,76.574989,2533.0,2118.501953,6.0360
2021-07-28,81.489998,6083.5,57.794998,686.842773,26483.312500,26.095192,23.327801,46.102501,19.260000,9.92375,927.608398,21.222771,106.037025,4.323393,4.802358,90.358589,76.574989,2531.5,2119.250977,6.4538


In [2]:
daily_returns = clean_prices.pct_change().dropna(how="all")

print("Returns shape:", daily_returns.shape)
daily_returns.head()

Returns shape: (1283, 20)


,VWRP.L,SWDA.L,VUAG.L,ISF.L,EQQQ.L,VEUR.L,VJPN.L,VFEG.L,IUIT.L,IUHC.L,INRG.L,VGOV.L,IGLO.L,IBTG.L,AGGG.L,CORP.L,GHYS.L,SGLN.L,IWDP.L,ICHN.AS
Date,,,,,,,,,,,,,,,,,,,,
2021-07-23,0.007149,0.009452,0.010567,0.008783,0.010788,0.008154,0.003848,-0.016443,0.010680,0.011877,-0.014195,-0.001749,-0.002156,0.000201,-0.001497,-0.000934,0.001532,-0.003899,0.001883,-0.038208
2021-07-26,-0.004610,-0.002931,-0.002228,-0.000290,-0.001455,-0.001716,-0.011978,-0.022150,-0.000387,-0.004291,-0.010400,0.001102,0.003544,-0.000101,0.002623,0.001590,0.000306,-0.005675,-0.003875,-0.041280
2021-07-27,-0.014749,-0.011432,-0.010308,-0.005371,-0.022088,-0.005810,-0.005140,-0.032683,-0.015857,0.002282,-0.022635,0.002950,0.002842,0.000101,0.001869,0.001773,-0.000204,-0.002952,0.000471,-0.067799
2021-07-28,0.008165,0.005039,0.003298,0.004232,0.012325,0.002799,0.002339,0.028156,0.009170,0.004047,0.023435,-0.001596,-0.001976,0.000151,-0.001306,-0.001304,0.000000,-0.000592,0.000354,0.069218
2021-07-29,0.000982,0.000740,-0.001817,0.009592,-0.003039,0.005171,0.001118,0.004772,0.002856,0.003905,0.008351,-0.000399,0.003959,-0.000050,0.003922,0.004759,0.001325,0.010468,-0.001649,0.006523


In [3]:
TRADING_DAYS = 252

annualised_volatility = (
    daily_returns.std() * np.sqrt(TRADING_DAYS)
)

volatility_table = (
    annualised_volatility
    .sort_values(ascending=False)
    .rename("Annualised_Volatility")
    .to_frame()
)

volatility_table["Annualised_Volatility_Percent"] = (
    volatility_table["Annualised_Volatility"] * 100
).round(2)

volatility_table

,Annualised_Volatility,Annualised_Volatility_Percent
INRG.L,0.309624,30.96
ICHN.AS,0.290806,29.08
IUIT.L,0.237614,23.76
SWDA.L,0.228266,22.83
EQQQ.L,0.192915,19.29
SGLN.L,0.165588,16.56
VJPN.L,0.157147,15.71
VFEG.L,0.152646,15.26
IUHC.L,0.148640,14.86
VUAG.L,0.142655,14.27


In [4]:
cumulative_growth = (1 + daily_returns).cumprod()
running_peak = cumulative_growth.cummax()

drawdowns = (
    cumulative_growth / running_peak
) - 1

maximum_drawdown = drawdowns.min()

drawdown_table = (
    maximum_drawdown
    .sort_values()
    .rename("Maximum_Drawdown")
    .to_frame()
)

drawdown_table["Maximum_Drawdown_Percent"] = (
    drawdown_table["Maximum_Drawdown"] * 100
).round(2)

drawdown_table

,Maximum_Drawdown,Maximum_Drawdown_Percent
INRG.L,-0.585248,-58.52
ICHN.AS,-0.512571,-51.26
VGOV.L,-0.373780,-37.38
IUIT.L,-0.334615,-33.46
IWDP.L,-0.302688,-30.27
EQQQ.L,-0.281650,-28.16
SWDA.L,-0.274175,-27.42
IGLO.L,-0.258825,-25.88
SGLN.L,-0.248895,-24.89
CORP.L,-0.248024,-24.80


In [5]:
CONFIDENCE_LEVEL = 0.95
tail_probability = 1 - CONFIDENCE_LEVEL

daily_var = daily_returns.quantile(tail_probability)

daily_cvar = daily_returns.apply(
    lambda returns: returns[
        returns <= returns.quantile(tail_probability)
    ].mean()
)

tail_risk_table = pd.DataFrame({
    "Daily_VaR_95": daily_var,
    "Daily_CVaR_95": daily_cvar
})

# Convert negative returns into positive loss percentages
tail_risk_table["Daily_VaR_95_Percent"] = (
    -tail_risk_table["Daily_VaR_95"] * 100
).round(2)

tail_risk_table["Daily_CVaR_95_Percent"] = (
    -tail_risk_table["Daily_CVaR_95"] * 100
).round(2)

tail_risk_table.sort_values(
    "Daily_CVaR_95_Percent",
    ascending=False
)

,Daily_VaR_95,Daily_CVaR_95,Daily_VaR_95_Percent,Daily_CVaR_95_Percent
INRG.L,-0.025208,-0.038356,2.52,3.84
ICHN.AS,-0.026270,-0.038307,2.63,3.83
IUIT.L,-0.024503,-0.032754,2.45,3.28
EQQQ.L,-0.019855,-0.027479,1.99,2.75
SGLN.L,-0.015873,-0.024499,1.59,2.45
SWDA.L,-0.013244,-0.022911,1.32,2.29
VJPN.L,-0.015502,-0.022332,1.55,2.23
VFEG.L,-0.015155,-0.022109,1.52,2.21
VUAG.L,-0.014310,-0.021214,1.43,2.12
IUHC.L,-0.014949,-0.021081,1.49,2.11


In [6]:
benchmark_ticker = "VWRP.L"
benchmark_returns = daily_returns[benchmark_ticker]

beta_values = {}

for ticker in daily_returns.columns:
    covariance = daily_returns[ticker].cov(benchmark_returns)
    benchmark_variance = benchmark_returns.var()

    beta_values[ticker] = covariance / benchmark_variance

beta_table = (
    pd.Series(beta_values, name="Beta")
    .sort_values(ascending=False)
    .to_frame()
)

beta_table["Beta"] = beta_table["Beta"].round(3)

beta_table

,Beta
IUIT.L,1.471
EQQQ.L,1.355
VUAG.L,1.062
INRG.L,1.040
VWRP.L,1.000
SWDA.L,0.968
ICHN.AS,0.940
VFEG.L,0.841
VEUR.L,0.840
VJPN.L,0.822


In [7]:
risk_metrics = pd.DataFrame({
    "Annualised_Volatility": annualised_volatility,
    "Maximum_Drawdown": maximum_drawdown,
    "Daily_VaR_95": daily_var,
    "Daily_CVaR_95": daily_cvar,
    "Beta": pd.Series(beta_values)
})

risk_metrics.index.name = "Ticker"

# Add easier-to-read percentage columns
risk_metrics["Annualised_Volatility_Percent"] = (
    risk_metrics["Annualised_Volatility"] * 100
).round(2)

risk_metrics["Maximum_Drawdown_Percent"] = (
    risk_metrics["Maximum_Drawdown"] * 100
).round(2)

risk_metrics["Daily_VaR_95_Percent"] = (
    -risk_metrics["Daily_VaR_95"] * 100
).round(2)

risk_metrics["Daily_CVaR_95_Percent"] = (
    -risk_metrics["Daily_CVaR_95"] * 100
).round(2)

risk_metrics["Beta"] = risk_metrics["Beta"].round(3)

risk_metrics.sort_values(
    "Annualised_Volatility",
    ascending=False
)

,Annualised_Volatility,Maximum_Drawdown,Daily_VaR_95,Daily_CVaR_95,Beta,Annualised_Volatility_Percent,Maximum_Drawdown_Percent,Daily_VaR_95_Percent,Daily_CVaR_95_Percent
Ticker,,,,,,,,,
INRG.L,0.309624,-0.585248,-0.025208,-0.038356,1.040,30.96,-58.52,2.52,3.84
ICHN.AS,0.290806,-0.512571,-0.026270,-0.038307,0.940,29.08,-51.26,2.63,3.83
IUIT.L,0.237614,-0.334615,-0.024503,-0.032754,1.471,23.76,-33.46,2.45,3.28
SWDA.L,0.228266,-0.274175,-0.013244,-0.022911,0.968,22.83,-27.42,1.32,2.29
EQQQ.L,0.192915,-0.281650,-0.019855,-0.027479,1.355,19.29,-28.16,1.99,2.75
SGLN.L,0.165588,-0.248895,-0.015873,-0.024499,0.037,16.56,-24.89,1.59,2.45
VJPN.L,0.157147,-0.183434,-0.015502,-0.022332,0.822,15.71,-18.34,1.55,2.23
VFEG.L,0.152646,-0.190722,-0.015155,-0.022109,0.841,15.26,-19.07,1.52,2.21
IUHC.L,0.148640,-0.176304,-0.014949,-0.021081,0.560,14.86,-17.63,1.49,2.11


In [8]:
etf_universe = pd.read_csv(
    "../data/processed/etf_universe_clean.csv"
)

risk_metrics_output = etf_universe[
    [
        "ETF_ID",
        "ETF_Name",
        "Ticker",
        "Provider",
        "Risk_Category",
        "Asset_Class"
    ]
].merge(
    risk_metrics.reset_index(),
    on="Ticker",
    how="left"
)

print(
    "ETFs missing risk metrics:",
    risk_metrics_output["Annualised_Volatility"].isna().sum()
)

risk_metrics_output.to_csv(
    "../data/processed/etf_risk_metrics.csv",
    index=False
)

risk_metrics_output.head()

ETFs missing risk metrics: 0


,ETF_ID,ETF_Name,Ticker,Provider,Risk_Category,Asset_Class,Annualised_Volatility,Maximum_Drawdown,Daily_VaR_95,Daily_CVaR_95,Beta,Annualised_Volatility_Percent,Maximum_Drawdown_Percent,Daily_VaR_95_Percent,Daily_CVaR_95_Percent
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Vanguard,Medium,Equity,0.128639,-0.176420,-0.012678,-0.018968,1.000,12.86,-17.64,1.27,1.90
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,iShares,Medium,Equity,0.228266,-0.274175,-0.013244,-0.022911,0.968,22.83,-27.42,1.32,2.29
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Vanguard,Medium,Equity,0.142655,-0.208796,-0.014310,-0.021214,1.062,14.27,-20.88,1.43,2.12
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L,iShares,Medium,Equity,0.125398,-0.131797,-0.012368,-0.019460,0.664,12.54,-13.18,1.24,1.95
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Invesco,Higher,Equity,0.192915,-0.281650,-0.019855,-0.027479,1.355,19.29,-28.16,1.99,2.75
